Imports


In [54]:
import numpy as np

In [55]:
N_INPUTS: int = 3
N_LAYERS: int = 3
N_NODESPERLAYER: int = 8
N_OUTPUTS: int = 1

rng = np.random.default_rng(42)

inWeights = rng.normal(0, np.sqrt(2 / (N_INPUTS)), [N_NODESPERLAYER, N_INPUTS])
inBias = rng.normal(0, np.sqrt(2 / (N_INPUTS)), N_NODESPERLAYER)
layerWeights = rng.normal(0, np.sqrt(2 / (N_INPUTS)), [N_LAYERS - 1, N_NODESPERLAYER, N_NODESPERLAYER])
layerBias = rng.normal(0, np.sqrt(2 / (N_INPUTS)), [N_LAYERS - 1, N_NODESPERLAYER])
outWeights = rng.normal(0, np.sqrt(2 / (N_INPUTS)), [N_OUTPUTS, N_NODESPERLAYER])
outBias = rng.normal(0, np.sqrt(2 / (N_INPUTS)))


def relu(x):
    return np.maximum(0, x)

# Forwarpass
def RunNetwork(_input, _inWeights, _inBias, _layerWeights, _layerBias, _outWeights, _outBias):
    act = relu(_inWeights @ _input + _inBias)  # input linear + relu
    for W, b in zip(_layerWeights, _layerBias):
        act = relu(W @ act + b)  # loop over layers: linear + relu
    act = _outWeights @ act + _outBias  # linear output layer
    return act


RunNetwork(rng.random(3), inWeights, inBias, layerWeights, layerBias, outWeights, outBias)

array([0.4960931])

In [56]:
class Tensor:
    def __init__(self, data: np.ndarray | list[float]):
        if type(data) == np.ndarray:
            self.data = data
        else:
            self.data = np.array(data, dtype=np.double)
        self.shape = np.shape(self.data)
        self.grad = np.zeros(self.shape)
        self._backward = lambda: None

    def __repr__(self):
        return f"Tensor(data={self.data})"

    def reshape(self, newShape):
        self.data.reshape(newShape)
        self.shape = np.shape(self.data)

    # --- Rechenoperationen ---
    # Komponentenweise Addition
    def __add__(self, other):
        out = Tensor(self.data + other.data)

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    # Matrix-Matrix Multiplikation
    def __matmul__(self, other):
        out = Tensor(self.data @ other.data)

        def _backward():
            self.grad += 0
            other.grad += 0

        out._backward = _backward
        return out

    # Komponentenweise Multiplikation
    def __mul__(self, other):
        out = Tensor(self.data * other.data)

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out


def LinTens(inTensor: Tensor | list[Tensor], schema: str) -> Tensor:
    tensors = [inTensor] if isinstance(inTensor, Tensor) else inTensor  # Normalize input tensors to list
    out = Tensor(np.einsum(schema, *[t.data for t in tensors]))  # Forwardpass

    split_schema = schema.split("->")
    in_index = split_schema[0].split(",")
    out_index = split_schema[1]

    # Identify missing indeces, *and their position*, in target. Remove missing from target. Kontract what is left.
    # Add 1D axes at missing spots. Broadcast new axes to desired length.

    def _backward():
        for idt, t in enumerate(tensors):
            domain: list[str] = [chars for i, chars in enumerate(in_index) if idt != i] + [
                out_index
            ]  # ordered list of index strings
            domain_set: set[str] = set("".join(domain))  # unique domain indeces

            missing: set[int] = {
                i for i, char in enumerate(out_index) if char not in domain_set
            }  # positions of indeces in out_index, that arent in domain

            kept: str = "".join(
                char for i, char in enumerate(in_index[idt]) if i not in missing
            )  # target without indeces, that dont occure in domain
            back_schema: str = ",".join(domain) + "->" + kept

            others: list[np.ndarray] = [tens.data for i, tens in enumerate(tensors) if idt != i]  # other tensors data

            g = np.einsum(back_schema, *others, out.grad)  # Contract the Tensor in all available indeces

            target = in_index[idt]
            for pos, c in enumerate(target):
                if c in missing:
                    g = np.expand_dims(g, pos)

            g = np.broadcast_to(g, t.data.shape)
            t.grad += g

    out._backward = _backward
    return out

In [57]:
x = np.arange(2)
y = np.expand_dims(x, (0,-1))
y = np.broadcast_to(y, (2,2,2))

print("shape x: \n", np.shape(x))
print("shape y: \n", np.shape(y))
print("y: \n", y)

def stretch(tensor: Tensor, insertion: tuple[int, ...], length: tuple[int, ...]) -> Tensor:
    array = np.expand_dims(tensor.data, insertion)

    shape = list(tensor.shape)
    for i, val in zip(insertion, length): 
        shape.insert(i, val)
    out = Tensor(np.broadcast_to(array, shape))
    return out




shape x: 
 (2,)
shape y: 
 (2, 2, 2)
y: 
 [[[0 0]
  [1 1]]

 [[0 0]
  [1 1]]]


In [ ]:
x = np.arange(9).reshape((3,3))

y = np